# TRUSTLENS AI — Phase 2.1: Multilingual Transformer (`google/muril-base-cased`) GPU Training Notebook

**Target Hardware:** Google Colab T4 GPU (16 GB VRAM) or Kaggle P100/T4  
**Dataset Version:** `2.0.5` (`text_train.parquet`, `text_val.parquet`, `text_test.parquet` — leak-free `StratifiedGroupKFold` splits, `random_seed=42`)  
**Classes:** `0: legitimate`, `1: spam`, `2: smishing`, `3: scam`

> Note: Do NOT run this notebook on local 16 GB RAM CPU-only hardware.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pyarrow
import torch, random, numpy as np, pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Load the exact Phase 2.0.5 leak-free Parquet splits
train_df = pd.read_parquet('data/processed/text_train.parquet')
val_df = pd.read_parquet('data/processed/text_val.parquet')
test_df = pd.read_parquet('data/processed/text_test.parquet')

LABEL_MAP = {'legitimate': 0, 'spam': 1, 'smishing': 2, 'scam': 3}
for df in (train_df, val_df, test_df):
    df['label_id'] = df['canonical_label'].map(LABEL_MAP).astype(int)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['cleaned_text'], truncation=True, padding='max_length', max_length=128)

ds_train = Dataset.from_pandas(train_df[['cleaned_text', 'label_id']].rename(columns={'label_id': 'labels'})).map(tokenize_fn, batched=True)
ds_val = Dataset.from_pandas(val_df[['cleaned_text', 'label_id']].rename(columns={'label_id': 'labels'})).map(tokenize_fn, batched=True)
ds_test = Dataset.from_pandas(test_df[['cleaned_text', 'label_id']].rename(columns={'label_id': 'labels'})).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='ml/artifacts/text_detection/transformer_muril',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
)

trainer.train()
test_metrics = trainer.evaluate(ds_test)
print('Held-Out Test Metrics:', test_metrics)